In [1]:
%matplotlib inline
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pathlib

from functools import lru_cache

import torch
import torchvision
from torchvision.models.detection import mask_rcnn, faster_rcnn
import tqdm
import transforms as T

**Load Data**

**Set learning parameters**

In [2]:
batch_size = 128
new_epochs = 10
lr = 0.001

**Data Control**

region detection

In [3]:
class DisplayDataset(torch.utils.data.Dataset):
    def __init__(self, background='synthetic/empty.png', digit_map='synthetic/digits.png', augment=False):
        # This should be an image of the display with all segments turned off
        self.background = torchvision.transforms.functional.convert_image_dtype(
            torchvision.io.read_image(background)[:3], torch.float32
        )
        # For every pixel this indicated which segment of which digit it corresponds to (or if it is part of the
        # background). The first segment of the first digit (left to right) will have index 1. With seven segments
        # for each digit the second digit will have indices 8-14, the third will have indices 15-21 and so on
        self.digit_map = torchvision.io.read_image(digit_map)[0]
        self.num_digits = self.digit_map.max().item() // 7

        # https://en.wikipedia.org/wiki/Seven-segment_display#Hexadecimal
        self.segment_encodings = (
            0b0111111,
            0b0000110,
            0b1011011,
            0b1001111,
            0b1100110,
            0b1101101,
            0b1111101,
            0b0000111,
            0b1111111,
            0b1101111,
        )

        # potentially add some variation to the data in terms of changing scale, jitter and overall appearance
        transforms = [T.ScaleJitter(target_size=(100, 200), scale_range=((0.8, 1.2) if augment else (1, 1)))]
        if augment:
            transforms.append(T.RandomPhotometricDistort())
        self.transform = T.Compose(transforms)

    def __len__(self):
        return 1_000_000

    def __getitem__(self, number):
        number = int(number)  # cast to int in case input is a torch.Tensor

        if not (0 <= number < 1_000_000):
            raise ValueError('only numbers in the range [0, 1_000_000) can be displayed')

        image = self.background.clone()
        height, width = image.shape[-2:]

        target = {
            'boxes': torch.zeros((self.num_digits, 4), dtype=torch.float32),
            'labels': torch.zeros(self.num_digits, dtype=torch.int64),
            'image_id': torch.tensor(number, dtype=torch.int64),
            'area': torch.zeros(self.num_digits, dtype=torch.float32),
            'iscrowd': torch.zeros(self.num_digits, dtype=torch.uint8),
            'masks': torch.zeros((self.num_digits, height, width), dtype=torch.uint8),
        }

        number_string = str(number).zfill(self.num_digits)
        for digit_index, digit in enumerate(number_string):
            target['labels'][digit_index] = int(digit) + 1

            digit_map = target['masks'][digit_index]
            for segment_index in range(7):
                visible = bool(self.segment_encodings[int(digit)] & (1 << segment_index))
                if visible:
                    digit_map[self.digit_map == (digit_index * 7 + segment_index + 1)] = 1

            coords_y, coords_x = torch.where(digit_map)
            target['boxes'][digit_index, 0] = coords_x.min()
            target['boxes'][digit_index, 1] = coords_y.min()
            target['boxes'][digit_index, 2] = coords_x.max()
            target['boxes'][digit_index, 3] = coords_y.max()

            target['area'][digit_index] = (coords_x.max() - coords_x.min()) * (coords_y.max() - coords_y.min())

        for digit_index in range(self.num_digits):
            mask = target['masks'][digit_index] == 1
            image[(mask)[None].expand(3, -1, -1)] = 0.1

        return self.transform(image, target)

In [4]:
indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-50])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-50:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, num_workers=4,
    collate_fn=utils.collate_fn)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test, batch_size=1, shuffle=False, num_workers=4,
    collate_fn=utils.collate_fn)

NameError: name 'dataset' is not defined

**Load Model**

In [8]:
@lru_cache(maxsize=None)
def load_model(model_path=None):
    num_classes = 11  # 10 digits + background

    model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = faster_rcnn.FastRCNNPredictor(in_features, num_classes)

    # replace the mask predictor with a new one
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = mask_rcnn.MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    if model_path is not None:
        model.load_state_dict(torch.load(model_path, map_location='cpu'))

    return model

**Train Data**

In [ ]:
dataset = DisplayDataset(augment=True)

random_indices = torch.randint(0, len(dataset), [1_000])
dataset_train = torch.utils.data.Subset(dataset, random_indices)

dataloader = torch.utils.data.DataLoader(
    dataset_train, batch_size=2, shuffle=True, num_workers=0, collate_fn=
)

model = load_model()
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

model.train()

for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
        # update the learning rate
        lr_scheduler.step()
        # evaluate on the test dataset
        evaluate(model, data_loader_test, device=device)

**Testing**